This reads both CSV files, finds the rows in `J1_OmegaMid.csv` where `verified == 0`, takes the corresponding cell IDs, and extracts those cells from `cell_def.csv`. The result is saved as `cell_def_failed.csv`.

In [3]:
import pandas as pd

cells = pd.read_csv("../inputs/cell_def.csv")
res = pd.read_csv("../results/J1_OmegaMid.csv")

cells[cells["i"].isin(res.loc[res["verified"] == 0, "cell_id"])].to_csv(
    "../inputs/cell_def_failed.csv", index=False
)

It reads `cell_def_failed.csv`, updates the specified column values, and saves it again with the same filename.

In [5]:
import pandas as pd

df = pd.read_csv("../inputs/cell_def_failed.csv")
df = df.assign(
    fem_order_lower_LG=3,
)
df.to_csv("../inputs/cell_def_failed.csv", index=False)

In [ ]:
import pandas as pd

base = pd.read_csv("../inputs/cell_def_origin_J1.csv").set_index("i")
failed = pd.read_csv("../inputs/cell_def_failed_J1.csv").set_index("i")

base.update(failed)
base.reset_index().to_csv("../inputs/cell_def.csv", index=False)

In [4]:
import pandas as pd

results = pd.read_csv("../results/J2_OmegaMid.csv")
cell_def = pd.read_csv("../inputs/cell_def_v3.csv")

target_ids = results.loc[results["verified"] == 0, "cell_id"]
cell_def.loc[cell_def["i"].isin(target_ids), "mesh_size_lower_cr"] = 0.001

cell_def.to_csv("../inputs/cell_def_v3.csv", index=False)

In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("../inputs/cell_def_v4.csv")
df = df[df["x_sup"] * np.tan(df["theta_sup"]) >= 0.0395].copy()
df["i"] = range(1, len(df) + 1)
df.to_csv("../inputs/cell_def_v4.csv", index=False)

In [9]:
import pandas as pd
import numpy as np

cells = pd.read_csv("../inputs/cell_def_v3.csv")
deleted = cells.loc[cells["x_sup"] * np.tan(cells["theta_sup"]) < 0.0395, "i"]

res = pd.read_csv("../results/J1_OmegaMid_v3.csv")
res = res[~res["cell_id"].isin(deleted)].copy()
res["cell_id"] = range(1, len(res) + 1)

res.to_csv("../results/J1_OmegaMid_v4.csv", index=False)

In [13]:
import csv
from decimal import Decimal

infile = "../inputs/cell_def_v3.csv"
outfile = "../inputs/cell_def_v4.csv"

# 文字列はそのまま保持して読む
with open(infile, newline="") as f:
    rows = list(csv.DictReader(f))
    fields = rows[0].keys()

# 端点は文字列を保持しつつ，順序付けだけ Decimal を使う
x_vals = sorted({r["x_inf"] for r in rows} | {r["x_sup"] for r in rows}, key=Decimal)
t_vals = sorted({r["theta_inf"] for r in rows} | {r["theta_sup"] for r in rows}, key=Decimal)
x_id = {s: i for i, s in enumerate(x_vals)}
t_id = {s: i for i, s in enumerate(t_vals)}

# 各セルが覆う最小格子片を求める
pieces = []
areas = []
for r in rows:
    xi, xs = x_id[r["x_inf"]], x_id[r["x_sup"]]
    ti, ts = t_id[r["theta_inf"]], t_id[r["theta_sup"]]
    pcs = {(i, j) for i in range(xi, xs) for j in range(ti, ts)}
    pieces.append(pcs)
    areas.append((Decimal(r["x_sup"]) - Decimal(r["x_inf"])) *
                 (Decimal(r["theta_sup"]) - Decimal(r["theta_inf"])))

# 各最小格子片が何個のセルで覆われているか
cover_count = {}
for pcs in pieces:
    for p in pcs:
        cover_count[p] = cover_count.get(p, 0) + 1

# 大きいセルから順に，ほかで完全に覆われていれば削除
keep = [True] * len(rows)
for k in sorted(range(len(rows)), key=lambda i: (-areas[i], i)):
    if all(cover_count[p] >= 2 for p in pieces[k]):
        keep[k] = False
        for p in pieces[k]:
            cover_count[p] -= 1

# 通し番号を振り直す（他の数値文字列は絶対に変更しない）
out_rows = []
for new_i, r in enumerate((rows[i] for i in range(len(rows)) if keep[i]), start=1):
    rr = dict(r)
    rr["i"] = str(new_i)
    out_rows.append(rr)

with open(outfile, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(out_rows)

In [24]:
import csv
from decimal import Decimal

cellfile = "../inputs/cell_def_v4.csv"
infile   = "../results/J1_OmegaMid.csv"
outfile  = "../results/J1_OmegaMid_v3.csv"

# --- 1. cell_def_v4.csv から「削除されずに残る元の i 」を求める ---
with open(cellfile, newline="") as f:
    cells = list(csv.DictReader(f))

x_vals = sorted({r["x_inf"] for r in cells} | {r["x_sup"] for r in cells}, key=Decimal)
t_vals = sorted({r["theta_inf"] for r in cells} | {r["theta_sup"] for r in cells}, key=Decimal)
x_id = {s: i for i, s in enumerate(x_vals)}
t_id = {s: i for i, s in enumerate(t_vals)}

pieces, areas = [], []
for r in cells:
    xi, xs = x_id[r["x_inf"]], x_id[r["x_sup"]]
    ti, ts = t_id[r["theta_inf"]], t_id[r["theta_sup"]]
    pcs = {(i, j) for i in range(xi, xs) for j in range(ti, ts)}
    pieces.append(pcs)
    areas.append((Decimal(r["x_sup"]) - Decimal(r["x_inf"])) *
                 (Decimal(r["theta_sup"]) - Decimal(r["theta_inf"])))

cover_count = {}
for pcs in pieces:
    for p in pcs:
        cover_count[p] = cover_count.get(p, 0) + 1

keep = [True] * len(cells)
for k in sorted(range(len(cells)), key=lambda i: (-areas[i], i)):
    if all(cover_count[p] >= 2 for p in pieces[k]):
        keep[k] = False
        for p in pieces[k]:
            cover_count[p] -= 1

old_to_new = {}
new_id = 1
for r, k in zip(cells, keep):
    if k:
        old_to_new[r["i"]] = str(new_id)
        new_id += 1

# --- 2. J1_OmegaMid_v3.csv から削除セルを落として cell_id を振り直す ---
with open(infile, newline="") as f:
    rows = list(csv.DictReader(f))
    fields = rows[0].keys()

out_rows = []
for r in rows:
    cid = r["cell_id"]
    if cid in old_to_new:
        rr = dict(r)
        rr["cell_id"] = old_to_new[cid]
        out_rows.append(rr)

with open(outfile, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(out_rows)

In [25]:
import csv

inp = "../inputs/cell_def_v4.csv"
out = "../inputs/cell_def_v5.csv"

with open(inp, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

for k, row in enumerate(rows, start=1):
    row["i"] = str(k)

with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)

In [30]:
import csv

# J1_OmegaMid.csv で cell_id == 0 のものを集める
with open("../results/J1_OmegaMid.csv", newline="") as f:
    ids = {r["cell_id"] for r in csv.DictReader(f) if r["cell_id"] == "0"}

# cell_def_v5.csv の対応する i の mesh_size_lower_LG を半分にする
with open("../inputs/cell_def_v3.csv", newline="") as f:
    rows = list(csv.DictReader(f))
    fieldnames = rows[0].keys()

for r in rows:
    if r["i"] in ids:
        r["mesh_size_lower_LG"] = 0.002

with open("../inputs/cell_def_v3.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)

In [12]:
import csv

path = "../inputs/cell_def_v3.csv"

with open(path, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

for row in rows:
    i = int(row["i"])
    if 69034 <= i <= 71481:
        row["mesh_size_lower_cr"] = "0.04"
        row["isLG"] = "1"
        row["mesh_size_lower_LG"] = "0.1249"
        row["fem_order_lower_LG"] = "3"

with open(path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)

In [22]:
import pandas as pd

results = pd.read_csv("../results/J2_OmegaMid.csv")
cell_def = pd.read_csv("../inputs/cell_def_v3.csv")

target_ids = results.loc[results["verified"] == 0, "cell_id"]
cell_def.loc[cell_def["i"].isin(target_ids), "fem_order_lower_LG"] = 3

cell_def.to_csv("../inputs/cell_def_v3.csv", index=False)

In [ ]:
import csv
from decimal import Decimal

cell_path = "../inputs/cell_def_v3.csv"
j1_path   = "../results/J1_OmegaMid.csv"   # 必要なら ../results/... に直す
j2_path   = "../results/J2_OmegaMid.csv"

target = 84035

def mid_str(a, b):
    x = (Decimal(a) + Decimal(b)) / Decimal("2")
    s = format(x, "f")
    s = s.rstrip("0").rstrip(".")
    return s if s else "0"

# --- 1,2. cell_def_v3.csv を更新 ---
with open(cell_path, newline="") as f:
    rows = list(csv.DictReader(f))
    fieldnames = rows[0].keys()

k = next(i for i, r in enumerate(rows) if int(r["i"]) == target)
r = rows[k]

x_inf, x_sup = r["x_inf"], r["x_sup"]
t_inf, t_sup = r["theta_inf"], r["theta_sup"]
x_mid = mid_str(x_inf, x_sup)
t_mid = mid_str(t_inf, t_sup)

base = {kk: vv for kk, vv in r.items() if kk != "i"}
new4 = [
    dict(base, i=str(target), x_inf=x_inf, x_sup=x_mid, theta_inf=t_inf, theta_sup=t_mid),
    dict(base, i=str(target+1), x_inf=x_mid, x_sup=x_sup, theta_inf=t_inf, theta_sup=t_mid),
    dict(base, i=str(target+2), x_inf=x_inf, x_sup=x_mid, theta_inf=t_mid, theta_sup=t_sup),
    dict(base, i=str(target+3), x_inf=x_mid, x_sup=x_sup, theta_inf=t_mid, theta_sup=t_sup),
]

rows = rows[:k] + new4 + rows[k+1:]

for i, r in enumerate(rows, start=1):
    r["i"] = str(i)

with open(cell_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(rows)

# --- 3. J1/J2 を更新 ---
def update_result(path):
    with open(path, newline="") as f:
        rows = list(csv.DictReader(f))
        fieldnames = rows[0].keys()

    for r in rows:
        cid = int(r["cell_id"])
        if cid >= target + 1:
            r["cell_id"] = str(cid + 3)

    conj = rows[0]["conjecture"]
    dummy = [
        {"conjecture": conj, "cell_id": str(target), "verified": "0", "J_lower": "", "status": "", "note": "", "run_timestamp": ""},
        {"conjecture": conj, "cell_id": str(target+1), "verified": "0", "J_lower": "", "status": "", "note": "", "run_timestamp": ""},
        {"conjecture": conj, "cell_id": str(target+2), "verified": "0", "J_lower": "", "status": "", "note": "", "run_timestamp": ""},
        {"conjecture": conj, "cell_id": str(target+3), "verified": "0", "J_lower": "", "status": "", "note": "", "run_timestamp": ""},
    ]
    rows += dummy
    rows.sort(key=lambda r: int(r["cell_id"]))

    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

update_result(j1_path)
update_result(j2_path)

In [1]:
start = 75058
infile = "../results_part1/J1_OmegaMid.csv"
outfile = "../results_part1/J1_OmegaMid.csv"

with open(infile, "r", encoding="utf-8", newline="") as f:
    lines = f.readlines()

with open(outfile, "w", encoding="utf-8", newline="") as f:
    f.write(lines[0])  # headerはそのまま
    for i, line in enumerate(lines[1:]):
        a, _, rest = line.split(",", 2)   # 1列目, 2列目, 残り
        f.write(f"{a},{start+i},{rest}")